# AI Resume Screener — NLP Analysis & Insights
**Author:** Aleena Anam | [GitHub](https://github.com/anam-aleena/ai-resume-screener)

This notebook demonstrates:
- TF-IDF keyword extraction from resumes and job descriptions
- Skill taxonomy matching and gap analysis
- Scoring methodology visualization
- Batch screening with ranked output

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from nlp_engine import extract_skills, keyword_frequency, compute_tfidf_similarity, clean_text

sns.set_theme(style='whitegrid')
%matplotlib inline
print('Imports OK')

## 1. TF-IDF Keyword Analysis

In [ ]:
jd = """
Senior Data Scientist — Python, SQL, scikit-learn, XGBoost, NLP,
statistical analysis, AWS, Docker, machine learning, deep learning,
feature engineering, predictive analytics, Pandas, NumPy.
3+ years experience required. Master's or PhD preferred.
"""

resume_strong = """
5 years Data Scientist. PhD Computer Science. Python, SQL, scikit-learn,
XGBoost, PyTorch, NLP, BERT, AWS, Docker, MLOps, Pandas, NumPy,
Matplotlib, statistical analysis, feature engineering, predictive analytics.
"""

resume_weak = """
Java developer 2 years. Spring Boot, REST APIs, MySQL, JavaScript.
Some Python experience. Bachelor in IT.
"""

print(f'JD → Strong Resume similarity: {compute_tfidf_similarity(resume_strong, jd):.3f}')
print(f'JD → Weak Resume similarity:   {compute_tfidf_similarity(resume_weak, jd):.3f}')

In [ ]:
# Visualize top JD keywords
jd_keywords = keyword_frequency(jd, top_n=15)
words, counts = zip(*jd_keywords)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(words, counts, color='#1a237e', edgecolor='white')
ax.set_title('Top Keywords in Job Description (TF-IDF)', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/jd_keywords.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Skill Taxonomy Matching

In [ ]:
skills_strong = extract_skills(resume_strong)
skills_weak   = extract_skills(resume_weak)
skills_jd     = extract_skills(jd)

print('JD requires skills:')
for cat, skills in skills_jd.items():
    if skills: print(f'  [{cat}]: {skills}')

print('\nStrong candidate has:')
for cat, skills in skills_strong.items():
    if skills: print(f'  [{cat}]: {skills}')

## 3. Full Batch Screening Demo

In [ ]:
import sys
sys.path.insert(0, '..')
from main import RESUMES, JOB_DESCRIPTION
from src.scorer import ResumeScorer

screener = ResumeScorer(JOB_DESCRIPTION, min_years_experience=3, min_education_score=3)
results  = screener.screen_batch(RESUMES)

# Build summary dataframe
rows = []
for r in results:
    rows.append({
        'Name':        r.candidate.name,
        'Total Score': r.scores.total_score,
        'Skill Match': r.scores.skill_match,
        'Experience':  r.scores.experience_match,
        'Education':   r.scores.education_match,
        'Decision':    r.scores.decision.replace('_',' ').title(),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Score comparison chart
metrics = ['Skill Match', 'Experience', 'Education']
x = np.arange(len(df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, metric in enumerate(metrics):
    ax.bar(x + i * width, df[metric], width, label=metric)

ax.set_xticks(x + width)
ax.set_xticklabels(df['Name'], rotation=15, ha='right')
ax.set_ylabel('Score')
ax.set_title('Score Comparison — All Candidates', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(y=65, color='gray', linestyle='--', alpha=0.5, label='Good threshold')
plt.tight_layout()
plt.savefig('../reports/score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()